# vepyr — Build Ensembl VEP Cache

Download an Ensembl VEP offline cache and convert it to optimized Parquet files and fjall KV stores.

In [1]:
import logging

logging.basicConfig(level=logging.INFO)

In [2]:
import os

os.environ["RUST_LOG"] = "info"

In [3]:
from vepyr import build_cache

In [ ]:
!pip install ipywidgets jupyter tqdm

In [4]:
# Configure
RELEASE = 115
CACHE_DIR = "/Users/mwiewior/workspace/data_vepyr"
SPECIES = "homo_sapiens"
ASSEMBLY = "GRCh38"
CACHE_TYPE = "vep"
# CACHE_TYPE = "merged"  # "vep", "merged", or "refseq"


# Set to an existing unpacked VEP cache directory to skip download, e.g.:
# LOCAL_CACHE = "/path/to/homo_sapiens/115_GRCh38"
# LOCAL_CACHE = "/Users/mwiewior/workspace/data_vepyr/homo_sapiens_merged/115_GRCh38"
LOCAL_CACHE = "/Users/mwiewior/workspace/data_vepyr/homo_sapiens_vep/115_GRCh38"

In [5]:
results = build_cache(
    release=RELEASE,
    cache_dir=CACHE_DIR,
    species=SPECIES,
    assembly=ASSEMBLY,
    cache_type=CACHE_TYPE,
    local_cache=LOCAL_CACHE,
    partitions=6,
)

INFO:vepyr:Using local cache: /Users/mwiewior/workspace/data_vepyr/homo_sapiens_vep/115_GRCh38
[2026-04-20T13:39:43Z INFO  datafusion_bio_function_vep::cache_builder] variation: all outputs exist, skipping (use overwrite to rebuild)
[2026-04-20T13:39:44Z INFO  datafusion_bio_function_vep::cache_builder] transcript: 25 main chroms, 1879 other contigs
[2026-04-20T13:41:53Z INFO  datafusion_bio_function_vep::cache_builder]   transcript: 532.2k rows [128.7s, 4.1k/s]
[2026-04-20T13:41:53Z INFO  datafusion_bio_function_vep::cache_builder] exon: parquet already exists, skipping (use overwrite to rebuild)
[2026-04-20T13:42:40Z INFO  datafusion_bio_function_vep::cache_builder] translation: chr1 core=22.8k sift=22.8k
[2026-04-20T13:43:17Z INFO  datafusion_bio_function_vep::cache_builder] translation: chr2 core=16.6k sift=16.6k
[2026-04-20T13:43:52Z INFO  datafusion_bio_function_vep::cache_builder] translation: chr3 core=15.1k sift=15.1k
[2026-04-20T13:44:13Z INFO  datafusion_bio_function_vep::ca

In [ ]:
import os

for path, rows in results:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    name = os.path.basename(path)
    print(f"{name:45s} {rows:>12,} rows  {size_mb:8.1f} MB")

In [ ]:
import pyarrow.parquet as pq

# Inspect one of the generated files
variation_file = [p for p, _ in results if "variation" in p][0]
table = pq.read_table(variation_file)
print(f"Schema: {table.schema}")
print(f"Rows: {table.num_rows:,}")
table.to_pandas().head()